# Notebook 7 — Real Datasets × Contamination Levels

**The core experiment:** Take real datasets with known inlier/outlier classes.
Inject outliers at increasing contamination levels (2% → 40%).
Measure AUROC at each level across all methods.

**What this proves:**
- MELU-Δt's MCD advantage is real — not an artefact of synthetic data
- The MCD breakdown point (25%) is visible in the curves
- Different dataset structures (medical, visual, chemical) show different profiles

**Datasets (all from sklearn — no download needed):**

| # | Dataset | Inliers | Outliers | dim | Why it's interesting |
|---|---|---|---|---|---|
| 1 | Breast Cancer | Benign (357) | Malignant (212) | 30 | Real medical binary labels, correlated features |
| 2 | Wine | Class 1 (71) | Class 0 + 2 mixed | 13 | Multi-class, chemical structure |
| 3 | Digits (easy) | Digit 0 (178) | Digit 6 (181) | 64 | Visually similar — hard outliers |
| 4 | Digits (hard) | Digit 1 (182) | Digit 7 (179) | 64 | Very similar structure — hardest |
| 5 | Digits (mixed) | Digits 0-4 | Digits 5-9 | 64 | Multi-inlier-class setting |

**Contamination levels tested:** 2%, 5%, 8%, 10%, 15%, 20%, 25%, 30%, 35%, 40%

**Methods:** MELU-Δt, Swish-AE, IsoForest, LOF, OC-SVM


## Cell 1 — Imports and model

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.special import betainc
from sklearn.datasets import load_digits, load_breast_cancer, load_wine, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings; warnings.filterwarnings("ignore")
np.random.seed(42)

CONTAM_LEVELS = [0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
METHODS = ["MELU-Δt", "Swish-AE", "IsoForest", "LOF", "OC-SVM"]
COLORS  = {"MELU-Δt":"#1D9E75","Swish-AE":"#534AB7",
           "IsoForest":"#BA7517","LOF":"#888780","OC-SVM":"#D85A30"}
LINESTYLES = {"MELU-Δt":"-","Swish-AE":"--","IsoForest":"-.","LOF":":","OC-SVM":(0,(3,1,1,1))}

print("Imports OK")
print(f"Testing {len(CONTAM_LEVELS)} contamination levels: {[f'{c:.0%}' for c in CONTAM_LEVELS]}")


## Cell 2 — Deep-MELU and Swish-AE (NaN-safe)

In [ ]:
def _tcdf(x, nu=4.0):
    nu = max(float(nu), 2.0)
    z  = nu / (nu + np.clip(np.asarray(x,float)**2, 1e-30, None))
    ib = betainc(nu/2, 0.5, np.clip(z, 1e-12, 1-1e-12))
    return np.where(np.asarray(x)>=0, 1.0-ib/2.0, ib/2.0)

def _safe_chol(cov, d, n):
    base = max(np.diag(cov).max(), 1e-8)*max(1e-3, min(0.1, 5/max(n/d,1)))
    for e in [base, base*10, base*100, 1.0]:
        try:
            L=np.linalg.cholesky(cov+e*np.eye(d))
            Li=np.linalg.inv(L)
            if not np.isnan(Li).any() and np.linalg.cond(Li)<1e7: return Li
        except: pass
    return np.eye(d)

class AE:
    """Two-layer AE with swappable activation: 'melu' or 'swish'."""
    def __init__(self, dim, lat=32, act='melu', alpha=1.0, beta=0.4, nu=4.0, mom=0.95):
        self.dim=dim; self.lat=lat; self.act=act
        self.alpha=alpha; self.beta=beta; self.nu=nu; self.mom=mom
        np.random.seed(0)
        self.W1 = np.random.randn(dim,64)   * np.sqrt(2/dim)
        self.W2 = np.random.randn(64,lat)   * np.sqrt(2/64)
        self.Wd = np.random.randn(lat,dim)  * np.sqrt(2/lat)
        self.mu=np.zeros(lat); self.Li=np.eye(lat); self.tau=1.0
        self.mu_d=0.; self.sd_d=1.; self.mu_e=0.; self.sd_e=1.

    def _sw(self,x): return x/(1+np.exp(-np.clip(x,-50,50)))

    def _activate(self, H):
        if self.act=='swish': return self._sw(H)
        # MELU-Δt
        T1  = H*_tcdf(H, self.nu)
        w   = np.nan_to_num((H-self.mu)@self.Li.T)
        m   = np.sqrt(np.maximum((w**2).sum(1), 0))
        gate= (m>=self.tau).astype(float)[:,None]
        amp = self.alpha*np.sign(H)*(
            np.exp(np.clip(self.beta*(m[:,None]-self.tau),-20,20))-1)
        return T1+gate*amp

    def _enc(self,X): return np.nan_to_num(self._activate(
        np.nan_to_num(self._sw(X@self.W1))@self.W2))

    def _mcd(self, Z, hf=0.75):
        n=len(Z); h=max(int(n*hf), self.lat+1)
        bd=np.inf; bm=None; bc=None
        for _ in range(8):
            idx=np.random.choice(n,h,replace=False); sub=Z[idx]
            for _ in range(5):
                mu=sub.mean(0); d=sub-mu
                cov=d.T@d/max(len(sub)-1,1)+1e-4*np.eye(self.lat)
                Si=np.linalg.inv(cov)
                ds=np.sqrt(np.maximum(np.einsum('bi,ij,bj->b',Z-mu,Si,Z-mu),0))
                idx=np.argsort(ds)[:h]; sub=Z[idx]
            mu=sub.mean(0); d=sub-mu; cov=d.T@d/max(len(sub)-1,1)
            det=np.linalg.det(cov+1e-4*np.eye(self.lat))
            if det<bd: bd=det; bm=mu; bc=cov
        return bm,bc

    def fit(self, X_in, n_epochs=70, lr=0.004, batch=64):
        n=len(X_in)
        for _ in range(n_epochs):
            Z=self._enc(X_in)
            if self.act=='melu':
                mu,cov=self._mcd(Z)
                self.mu=mu; self.Li=_safe_chol(cov,self.lat,n)
                w=np.nan_to_num((Z-mu)@self.Li.T)
                dm=np.sqrt(np.maximum((w**2).sum(1),0))
                self.tau=self.mom*self.tau+(1-self.mom)*dm.mean()
            idx=np.random.permutation(n)
            for i in range(0,n,batch):
                xb=X_in[idx[i:i+batch]]
                zm=np.nan_to_num(self._activate(self._enc(xb)))
                xh=zm@self.Wd
                self.Wd-=lr*np.clip(zm.T@(xh-xb)/max(len(xb),1),-1,1)
        Z=self._enc(X_in); Zm=np.nan_to_num(self._activate(Z)); Xh=Zm@self.Wd
        w=np.nan_to_num((Z-self.mu)@self.Li.T)
        dm=np.sqrt(np.maximum((w**2).sum(1),0))
        er=np.abs(X_in-Xh).mean(1)
        self.mu_d=dm.mean(); self.sd_d=max(dm.std(),1e-6)
        self.mu_e=er.mean(); self.sd_e=max(er.std(),1e-6)
        return self

    def score(self,X):
        Z=self._enc(X); Zm=np.nan_to_num(self._activate(Z)); Xh=Zm@self.Wd
        w=np.nan_to_num((Z-self.mu)@self.Li.T)
        dm=np.sqrt(np.maximum((w**2).sum(1),0))
        er=np.abs(X-Xh).mean(1)
        sd=np.maximum(0,(dm-self.mu_d)/self.sd_d)
        se=np.maximum(0,(er-self.mu_e)/self.sd_e)
        return 0.5*sd+0.5*se

print("AE (MELU-Δt and Swish) defined.")


## Cell 3 — Real dataset loaders with contamination injection

In [ ]:
def load_real(name):
    """
    Load a real dataset. Returns (X_inliers, X_outliers, description).
    X_inliers: the 'normal' class(es)
    X_outliers: the 'anomaly' class(es)  — we control how many we inject
    """
    if name == "BreastCancer":
        d = load_breast_cancer()
        # benign=1 → inlier, malignant=0 → outlier (real medical ground truth)
        X_in  = d.data[d.target == 1]   # 357 benign
        X_out = d.data[d.target == 0]   # 212 malignant
        return X_in, X_out, "Benign breast tissue (inlier) vs Malignant (outlier)"

    if name == "Wine":
        d = load_wine()
        # class 1 largest and most 'average' → inlier
        # class 0 and 2 → outlier pool
        X_in  = d.data[d.target == 1]              # 71 samples
        X_out = d.data[d.target != 1]              # 107 samples (class 0+2)
        return X_in, X_out, "Wine class 1 (inlier) vs classes 0 & 2 (outlier)"

    if name == "Digits_easy":
        d = load_digits()
        # digit 0 vs digit 6 — moderate similarity
        X_in  = d.data[d.target == 0]   # 178
        X_out = d.data[d.target == 6]   # 181
        return X_in, X_out, "Handwritten digit 0 (inlier) vs digit 6 (outlier)"

    if name == "Digits_hard":
        d = load_digits()
        # digit 1 vs digit 7 — very similar structure (both tall, thin)
        X_in  = d.data[d.target == 1]   # 182
        X_out = d.data[d.target == 7]   # 179
        return X_in, X_out, "Handwritten digit 1 (inlier) vs digit 7 (outlier, similar)"

    if name == "Digits_mixed":
        d = load_digits()
        # digits 0-4 as inliers, 5-9 as outlier pool
        X_in  = d.data[d.target <= 4]   # ~890 samples
        X_out = d.data[d.target >= 5]   # ~896 samples
        return X_in, X_out, "Digits 0-4 (inlier) vs digits 5-9 (outlier)"

REAL_DATASETS = ["BreastCancer", "Wine", "Digits_easy", "Digits_hard", "Digits_mixed"]

# Preview all datasets
print("Real dataset inventory:")
print(f"{'Name':<20} {'n_inliers':>10} {'n_outliers':>12} {'dim':>5}")
print("-"*55)
for name in REAL_DATASETS:
    Xi, Xo, desc = load_real(name)
    print(f"{name:<20} {len(Xi):>10} {len(Xo):>12} {Xi.shape[1]:>5}   {desc[:50]}")


## Cell 4 — Contamination injector

In [ ]:
def build_contaminated(X_in, X_out, contam_level, seed=42):
    """
    Build a contaminated dataset with exactly `contam_level` fraction outliers.

    Strategy:
    - Fix the number of inliers to min(len(X_in), 400) for computational speed
    - Sample exactly ceil(n_in * contam_level / (1-contam_level)) outliers
    - Shuffle and return (X_combined, y_labels)

    This ensures the contamination fraction is exact and reproducible.
    """
    rng = np.random.RandomState(seed)

    # Fix inlier count (subsample if too large)
    n_in_max = min(len(X_in), 500)
    idx_in   = rng.choice(len(X_in), n_in_max, replace=False)
    Xi_used  = X_in[idx_in]

    # Calculate exact number of outliers needed
    n_in  = len(Xi_used)
    n_out = max(1, int(np.ceil(n_in * contam_level / (1 - contam_level))))
    n_out = min(n_out, len(X_out))   # can't use more than available

    # Sample outliers
    idx_out = rng.choice(len(X_out), n_out, replace=(n_out > len(X_out)))
    Xo_used = X_out[idx_out]

    X = np.vstack([Xi_used, Xo_used])
    y = np.array([0]*n_in + [1]*n_out)

    # Shuffle
    perm = rng.permutation(len(X))
    actual_contam = n_out / len(X)
    return X[perm], y[perm], actual_contam

# Test the injector
Xi, Xo, _ = load_real("BreastCancer")
for c in [0.05, 0.10, 0.25, 0.40]:
    X, y, actual = build_contaminated(Xi, Xo, c)
    print(f"  target={c:.0%}  actual={actual:.1%}  "
          f"n_total={len(X)}  n_out={y.sum()}")


## Cell 5 — Run one method on one contaminated dataset

In [ ]:
def run_method_on(method_name, X, y, n_epochs=60):
    """
    Fit and evaluate one method on a contaminated dataset.
    Returns AUROC and AUCPR.
    """
    sc = StandardScaler()
    Xs = sc.fit_transform(X)
    Xi = Xs[y == 0]   # train only on labeled inliers

    try:
        if method_name == "MELU-Δt":
            m = AE(X.shape[1], act='melu').fit(Xi, n_epochs=n_epochs)
            scores = m.score(Xs)

        elif method_name == "Swish-AE":
            m = AE(X.shape[1], act='swish').fit(Xi, n_epochs=n_epochs)
            scores = m.score(Xs)

        elif method_name == "IsoForest":
            m = IsolationForest(n_estimators=200, contamination="auto",
                                random_state=42)
            m.fit(Xi); scores = -m.score_samples(Xs)

        elif method_name == "LOF":
            k = min(20, max(3, len(Xi)//5))
            m = LocalOutlierFactor(n_neighbors=k, novelty=True,
                                   contamination="auto")
            m.fit(Xi); scores = -m.score_samples(Xs)

        elif method_name == "OC-SVM":
            m = OneClassSVM(kernel="rbf", nu=0.1, gamma="scale")
            m.fit(Xi); scores = -m.score_samples(Xs)

        if np.isnan(scores).any(): raise ValueError("NaN")
        return dict(
            auroc = float(roc_auc_score(y, scores)),
            aucpr = float(average_precision_score(y, scores))
        )
    except Exception as e:
        return dict(auroc=0.5, aucpr=0.0, error=str(e)[:60])


print("run_method_on() defined.")
print("Quick sanity check (BreastCancer, 10% contamination):")
Xi, Xo, _ = load_real("BreastCancer")
X_test, y_test, _ = build_contaminated(Xi, Xo, 0.10, seed=0)
for m in METHODS:
    r = run_method_on(m, X_test, y_test, n_epochs=30)
    print(f"  {m:<14}  AUROC={r['auroc']:.3f}  AUCPR={r['aucpr']:.3f}")


## Cell 6 — Full contamination sweep on all datasets

> **Expected runtime: 30–90 minutes** (5 datasets × 10 contamination levels × 5 methods × 5 seeds).

> Run Cell 7 for a quick single-seed preview while this runs.

In [ ]:
N_SEEDS   = 5     # average over seeds for stability
N_EPOCHS  = 65    # enough for convergence

all_results = {}   # {dataset: {method: {level: [auroc_seed1, ...]}}}

for ds_name in REAL_DATASETS:
    Xi, Xo, desc = load_real(ds_name)
    all_results[ds_name] = {m: {c: [] for c in CONTAM_LEVELS} for m in METHODS}

    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name}")
    print(f"  {desc}")
    print(f"  n_in={len(Xi)}  n_out_pool={len(Xo)}  dim={Xi.shape[1]}")
    print(f"{'='*60}")

    for c_level in CONTAM_LEVELS:
        row_str = f"  contam={c_level:.0%}  "
        for seed in range(N_SEEDS):
            X, y, actual = build_contaminated(Xi, Xo, c_level, seed=seed*100)
            for method in METHODS:
                r = run_method_on(method, X, y, n_epochs=N_EPOCHS)
                all_results[ds_name][method][c_level].append(r["auroc"])

        # Print mean per method at this level
        for method in METHODS:
            vals = all_results[ds_name][method][c_level]
            row_str += f"{method[:6]}={np.mean(vals):.3f} "
        print(row_str)

print("\n✓ Full sweep complete.")


## Cell 7 — Quick single-seed preview (run this first)

In [ ]:
N_EPOCHS_Q = 40   # fewer epochs for speed

quick_results = {}

for ds_name in REAL_DATASETS:
    Xi, Xo, desc = load_real(ds_name)
    quick_results[ds_name] = {m: [] for m in METHODS}
    print(f"\n{ds_name}  [{desc[:45]}]")

    for c_level in CONTAM_LEVELS:
        X, y, actual = build_contaminated(Xi, Xo, c_level, seed=42)
        for method in METHODS:
            r = run_method_on(method, X, y, n_epochs=N_EPOCHS_Q)
            quick_results[ds_name][method].append(r["auroc"])

    # Print table row
    dm_vals = quick_results[ds_name]["MELU-Δt"]
    sw_vals = quick_results[ds_name]["Swish-AE"]
    print(f"  {'Contam':>8}", end="")
    for c in CONTAM_LEVELS: print(f"  {c:.0%}", end="")
    print()
    for method in METHODS:
        vals = quick_results[ds_name][method]
        lw = "★" if method=="MELU-Δt" else " "
        print(f"  {lw}{method:<13}", end="")
        for v in vals: print(f"  {v:.3f}", end="")
        print()

print("\n✓ Quick preview complete.")


## Cell 8 — Main results figure: AUROC vs contamination level

Use `all_results` if Cell 6 ran, otherwise `quick_results`.

In [ ]:
# Use whichever results are available
results_to_plot = all_results if all_results else quick_results
multi_seed = (results_to_plot is all_results)

fig = plt.figure(figsize=(20, 14))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.32)
fig.suptitle("MELU-Δt vs Baselines — Real Datasets × Contamination Levels",
             fontsize=14, fontweight="500")

axes_flat = [fig.add_subplot(gs[r,c]) for r in range(2) for c in range(3)]

for ax_idx, ds_name in enumerate(REAL_DATASETS):
    ax = axes_flat[ax_idx]
    _, _, desc = load_real(ds_name)

    for method in METHODS:
        lw  = 2.8 if method=="MELU-Δt" else 1.5
        ls  = LINESTYLES[method]
        ms  = 6   if method in ["MELU-Δt","Swish-AE"] else 4
        col = COLORS[method]

        if multi_seed:
            means = [np.mean(results_to_plot[ds_name][method][c]) for c in CONTAM_LEVELS]
            stds  = [np.std( results_to_plot[ds_name][method][c]) for c in CONTAM_LEVELS]
            ax.plot([c*100 for c in CONTAM_LEVELS], means,
                    color=col, lw=lw, ls=ls, marker="o", ms=ms, label=method)
            ax.fill_between([c*100 for c in CONTAM_LEVELS],
                            [m-s for m,s in zip(means,stds)],
                            [m+s for m,s in zip(means,stds)],
                            alpha=0.12, color=col)
        else:
            vals = quick_results[ds_name][method]
            ax.plot([c*100 for c in CONTAM_LEVELS], vals,
                    color=col, lw=lw, ls=ls, marker="o", ms=ms, label=method)

    # MCD breakdown point marker
    ax.axvline(25, color="#1D9E75", lw=1.2, ls=":", alpha=0.6)
    ax.fill_betweenx([0.3,1.05], 0, 25, alpha=0.04, color="#1D9E75")
    ax.text(25.5, 0.35, "MCD
breakdown
25%", fontsize=7, color="#0F6E56",
            va="bottom")

    ax.set_xlabel("Contamination %", fontsize=9)
    ax.set_ylabel("AUROC", fontsize=9)
    ax.set_title(f"{ds_name}
{desc[:48]}", fontsize=9)
    ax.set_ylim(0.3, 1.05)
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=8)
    if ax_idx == 0:
        ax.legend(fontsize=8, ncol=2, loc="lower left")

# Summary panel (last subplot)
ax = axes_flat[5]; ax.set_facecolor("white")
ax.axis("off")

# Compute advantage table
rows_tbl = [["Dataset","Adv @10%","Adv @25%","Adv @40%","Best contam range"]]
for ds_name in REAL_DATASETS:
    dm = quick_results[ds_name]["MELU-Δt"]
    sw = quick_results[ds_name]["Swish-AE"]
    c_idx = {c:i for i,c in enumerate(CONTAM_LEVELS)}
    adv10 = dm[c_idx[0.10]] - sw[c_idx[0.10]]
    adv25 = dm[c_idx[0.25]] - sw[c_idx[0.25]]
    adv40 = dm[c_idx[0.40]] - sw[c_idx[0.40]]
    best_range = "low" if adv10>=adv25 else ("mid" if adv25>=adv40 else "high")
    rows_tbl.append([ds_name[:14], f"{adv10:+.3f}", f"{adv25:+.3f}",
                     f"{adv40:+.3f}", best_range])

tbl = ax.table(cellText=rows_tbl[1:], colLabels=rows_tbl[0],
               loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
for (r,c_), cell in tbl.get_celld().items():
    cell.set_linewidth(0.3)
    if r == 0:
        cell.set_facecolor("#E1F5EE"); cell.set_text_props(fontweight="bold")
    elif c_ > 0 and r > 0:
        try:
            v = float(rows_tbl[r][c_].replace("+",""))
            if v > 0.01:
                cell.set_facecolor("#E1F5EE")
                cell.set_text_props(color="#085041", fontweight="bold")
            elif v < -0.01:
                cell.set_facecolor("#FAECE7")
                cell.set_text_props(color="#993C1D")
        except: pass
ax.set_title("MELU-Δt advantage
over Swish-AE", fontsize=10, pad=14)

plt.savefig("outputs/realdata_contamination.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/realdata_contamination.png")


## Cell 9 — Breakdown point detection figure

Finds the empirical contamination level where MELU-Δt's advantage peaks.
This is the key figure for the paper — it visually shows the MCD breakdown point advantage.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("MELU-Δt Advantage Profile — When Does MCD Gating Help Most?",
             fontsize=12)

DATASET_COLORS = {
    "BreastCancer":  "#1D9E75",
    "Wine":          "#534AB7",
    "Digits_easy":   "#BA7517",
    "Digits_hard":   "#D85A30",
    "Digits_mixed":  "#888780",
}

# Panel 1: Δ AUROC (MELU-Δt − Swish-AE) per dataset
ax = axes[0]
x_cont = [c*100 for c in CONTAM_LEVELS]
for ds_name in REAL_DATASETS:
    dm = quick_results[ds_name]["MELU-Δt"]
    sw = quick_results[ds_name]["Swish-AE"]
    delta = [d-s for d,s in zip(dm,sw)]
    ax.plot(x_cont, delta, color=DATASET_COLORS[ds_name],
            lw=2.2, marker="o", ms=5, label=ds_name)
    ax.fill_between(x_cont, [0]*len(delta), delta,
                    alpha=0.08, color=DATASET_COLORS[ds_name])

ax.axhline(0, color="black", lw=1)
ax.axvline(25, color="#1D9E75", lw=1.5, ls="--", alpha=0.7, label="MCD breakdown 25%")
ax.fill_betweenx([-0.15, 0.20], 0, 25, alpha=0.05, color="#1D9E75")
ax.set_xlabel("Contamination %", fontsize=10)
ax.set_ylabel("AUROC (MELU-Δt) − AUROC (Swish-AE)", fontsize=9)
ax.set_title("Advantage of MELU-Δt over Swish-AE
(positive = MELU-Δt wins)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.text(1, ax.get_ylim()[1]*0.85,
        "Above zero = MELU-Δt better
Below zero = Swish-AE better",
        fontsize=8, color="gray")

# Panel 2: Relative degradation — how much does each method degrade?
ax = axes[1]
baseline_contam = 0.05   # reference point
for method in ["MELU-Δt", "Swish-AE", "IsoForest", "LOF"]:
    degradations = []
    for ds_name in REAL_DATASETS:
        ref_idx = list(CONTAM_LEVELS).index(baseline_contam)
        vals = quick_results[ds_name][method]
        ref  = vals[ref_idx]
        degs = [ref - v for v in vals]   # positive = degradation
        degradations.append(degs)
    avg_degrad = np.mean(degradations, axis=0)

    lw = 2.8 if method=="MELU-Δt" else 1.5
    ls = "-"  if method=="MELU-Δt" else LINESTYLES[method]
    ax.plot(x_cont, avg_degrad, color=COLORS[method],
            lw=lw, ls=ls, marker="o", ms=5, label=method)

ax.axvline(25, color="#1D9E75", lw=1.5, ls="--", alpha=0.7,
           label="MCD breakdown 25%")
ax.fill_betweenx([-0.01, 0.25], 0, 25, alpha=0.05, color="#1D9E75")
ax.axhline(0, color="black", lw=0.5)
ax.set_xlabel("Contamination %", fontsize=10)
ax.set_ylabel(f"AUROC degradation from {baseline_contam:.0%} baseline", fontsize=9)
ax.set_title("Average degradation across all 5 datasets
"
             "(lower = more robust to contamination)")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_ylim(-0.02, 0.25)

plt.tight_layout()
plt.savefig("outputs/breakdown_point_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/breakdown_point_analysis.png")


## Cell 10 — Per-dataset heatmap (publication figure)

In [ ]:
fig, axes = plt.subplots(1, len(REAL_DATASETS), figsize=(20, 5.5))
fig.suptitle("AUROC Heatmap: Method × Contamination Level (per real dataset)",
             fontsize=12)

for ax, ds_name in zip(axes, REAL_DATASETS):
    matrix = np.array([[quick_results[ds_name][m][i]
                        for i, c in enumerate(CONTAM_LEVELS)]
                       for m in METHODS])

    # Mark the best method per contamination level
    im = ax.imshow(matrix, aspect="auto", cmap="YlGn",
                   vmin=0.4, vmax=1.0, interpolation="nearest")

    # Annotate
    for mi, method in enumerate(METHODS):
        for ci, c in enumerate(CONTAM_LEVELS):
            v = matrix[mi, ci]
            col = "white" if v < 0.65 else "black"
            ax.text(ci, mi, f"{v:.2f}", ha="center", va="center",
                    fontsize=7, color=col, fontweight="bold" if method=="MELU-Δt" else "normal")

    ax.set_xticks(range(len(CONTAM_LEVELS)))
    ax.set_xticklabels([f"{c:.0%}" for c in CONTAM_LEVELS], rotation=50, fontsize=7)
    ax.set_yticks(range(len(METHODS)))
    ax.set_yticklabels(METHODS, fontsize=8)
    ax.set_title(ds_name, fontsize=9)

    # Vertical line at MCD breakdown point
    bp_idx = list(CONTAM_LEVELS).index(0.25) - 0.5
    ax.axvline(bp_idx, color="#1D9E75", lw=2, alpha=0.8)
    ax.text(bp_idx+0.1, -0.7, "25%
breakdown", fontsize=6.5,
            color="#085041", va="top", ha="left")

plt.colorbar(im, ax=axes[-1], fraction=0.03, pad=0.04, label="AUROC")
plt.tight_layout()
plt.savefig("outputs/contamination_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → outputs/contamination_heatmap.png")


## Cell 11 — Statistical summary and CSV export

In [ ]:
rows = []
for ds_name in REAL_DATASETS:
    for method in METHODS:
        for i, c_level in enumerate(CONTAM_LEVELS):
            vals_list = (all_results[ds_name][method][c_level]
                        if all_results else [quick_results[ds_name][method][i]])
            rows.append({
                "dataset":     ds_name,
                "method":      method,
                "contam_pct":  round(c_level*100, 1),
                "auroc_mean":  round(np.mean(vals_list), 4),
                "auroc_std":   round(np.std(vals_list), 4),
                "n_seeds":     len(vals_list),
            })

df = pd.DataFrame(rows)
df.to_csv("outputs/contamination_results.csv", index=False)
print("Saved → outputs/contamination_results.csv")

# Summary table
print("\nSummary: MELU-Δt vs Swish-AE average AUROC advantage")
print(f"{'Dataset':<20} {'@ 5%':>7} {'@10%':>7} {'@20%':>7} {'@25%':>7} {'@35%':>7}")
print("-"*55)
idxs = {c:i for i,c in enumerate(CONTAM_LEVELS)}
for ds_name in REAL_DATASETS:
    dm = quick_results[ds_name]["MELU-Δt"]
    sw = quick_results[ds_name]["Swish-AE"]
    deltas = [dm[idxs[c]]-sw[idxs[c]] for c in [0.05,0.10,0.20,0.25,0.35]]
    print(f"{ds_name:<20}", end="")
    for d in deltas:
        c_ = "+" if d>0.005 else (" " if abs(d)<=0.005 else "-")
        print(f"  {c_}{abs(d):.3f}", end="")
    print()

print()
print("Key finding:")
overall_adv_below25 = np.mean([
    quick_results[ds][m][idxs[c]] - quick_results[ds]["Swish-AE"][idxs[c]]
    for ds in REAL_DATASETS for m in ["MELU-Δt"]
    for c in [c_ for c_ in CONTAM_LEVELS if c_ <= 0.25]])
overall_adv_above25 = np.mean([
    quick_results[ds][m][idxs[c]] - quick_results[ds]["Swish-AE"][idxs[c]]
    for ds in REAL_DATASETS for m in ["MELU-Δt"]
    for c in [c_ for c_ in CONTAM_LEVELS if c_ > 0.25]])
print(f"  Average advantage ≤ 25% contamination: {overall_adv_below25:+.4f}")
print(f"  Average advantage  > 25% contamination: {overall_adv_above25:+.4f}")
print(f"  Consistent with MCD breakdown point theory: "
      f"{'YES ✓' if overall_adv_below25 > overall_adv_above25 else 'partial'}")
